# 4 · Compose LCEL workflows

Use prompt templates, JSON/text parsers, `RunnableLambda`, `RunnableParallel`, and `RunnablePassthrough.assign`. These are runnable compositions, rather than the deprecated `LLMChain` / `SequentialChain` classes listed in the original outline. Configure the root `.env` before running Azure cells. Requests may incur charges. Stored outputs are cleared; no new live Azure results are supplied.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT))
from lab.config import get_chat_model

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
llm = get_chat_model()

## One prompt → parsed JSON
Extract a request's stated purpose and constraints. The example does not select a loan product or determine eligibility.

In [ ]:
intent_prompt = ChatPromptTemplate.from_template(
    "Extract the stated loan purpose and constraints from this message: {message}. "
    "Return only JSON with keys purpose and constraints. Do not infer missing values.")
intent_chain = intent_prompt | llm | JsonOutputParser()
print(intent_chain.invoke({"message": "I need a car loan for three years with an interest rate below nine percent."}))

## Sequential workflow: insurance intent → catalog category
The catalog contains illustrative names and IDs only. It does not contain prices, eligibility rules, benefits, or actual product terms. Category selection is deterministic after parsing.

In [ ]:
catalog = {
    "health": {"name": "Health Insurance", "id": "INS001"},
    "life": {"name": "Life Insurance", "id": "INS002"},
    "auto": {"name": "Auto Insurance", "id": "INS003"},
    "home": {"name": "Home Insurance", "id": "INS004"},
    "travel": {"name": "Travel Insurance", "id": "INS005"},
    "pet": {"name": "Pet Insurance", "id": "INS006"},
    "disability": {"name": "Disability Insurance", "id": "INS007"},
    "business": {"name": "Business Insurance", "id": "INS008"},
    "liability": {"name": "Liability Insurance", "id": "INS009"},
    "critical_illness": {"name": "Critical Illness Insurance", "id": "INS010"},
}
insurance_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract insurance intent only. Return JSON with category and constraints. "
     "Use a category from: " + ", ".join(catalog) + ", or unknown. Loan-only requests are unknown. "
     "Do not infer product benefits, prices, or eligibility."),
    ("human", "{message}"),
])
insurance_chain = insurance_prompt | llm | JsonOutputParser()
def select_product(intent):
    if not isinstance(intent, dict):
        raise ValueError("Intent must be a JSON object.")
    return catalog.get(intent.get("category"), {"name": "No matching category", "id": None})
sequential = insurance_chain | RunnableLambda(select_product)
print(sequential.invoke({"message": "I need insurance for my new car."}))

## Parallel inputs and named outputs
Intent extraction and language identification can run independently. Later stages receive parsed values, not stringified `AIMessage` objects. A failed stage stops the composition and should be inspected before retrying.

In [ ]:
language_prompt = ChatPromptTemplate.from_template("Identify the language of this message. Return only its language name: {message}")
language_chain = language_prompt | llm | StrOutputParser()
followup_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write one short follow-up in the requested language. Mention only the selected catalog name and ID. "
     "Ask the user to clarify needs; do not promise benefits, approval, pricing, or eligibility. "
     "If there is no matching category, ask for clarification."),
    ("human", "Category: {product}\nLanguage: {language}"),
])
full_chain = (
    RunnableParallel(intent=insurance_chain, language=language_chain)
    | RunnablePassthrough.assign(product=RunnableLambda(lambda x: select_product(x["intent"])))
    | RunnablePassthrough.assign(followup=followup_prompt | llm | StrOutputParser())
)
result = full_chain.invoke({"message": "I need insurance for my new car, preferably with a low premium."})
print(json.dumps(result, indent=2, ensure_ascii=False))

Try a loan-only message and inspect whether the model returns `unknown`. The JSON parser validates JSON syntax, not intent accuracy. This notebook does not implement router chains, underwriting, financial recommendations, or measured classification performance.